## Packages & Data

In [1]:
# Packages 
import polars as pl
import matplotlib.pyplot as plt

In [2]:
# Load data
df = pl.scan_csv("../auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [3]:
df.show(5)

time,src_user,dest_user,src_comp,dest_comp,auth_type,logon_type,auth_orientation,outcome
i64,str,str,str,str,str,str,str,str
1,"""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C1250""","""C586""","""NTLM""","""Network""","""LogOn""","""Success"""
1,"""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C586""","""C586""","""?""","""Network""","""LogOff""","""Success"""
1,"""C101$@DOM1""","""C101$@DOM1""","""C988""","""C988""","""?""","""Network""","""LogOff""","""Success"""
1,"""C1020$@DOM1""","""SYSTEM@C1020""","""C1020""","""C1020""","""Negotiate""","""Service""","""LogOn""","""Success"""
1,"""C1021$@DOM1""","""C1021$@DOM1""","""C1021""","""C625""","""Kerberos""","""Network""","""LogOn""","""Success"""


## Auth counts

In [ ]:
# Event Count
df.select(pl.len()).collect().item()

In [ ]:
# Human user counts
src_users = (
    df
    .filter(pl.col("src_user").str.starts_with("U"))
    .select(pl.col("src_user").unique())
    .collect()
    .to_series()
)

dest_users = (
    df
    .filter(pl.col("dest_user").str.starts_with("U"))
    .select(pl.col("dest_user").unique())
    .collect()
    .to_series()
)

all_users = set(src_users.to_list()) | set(dest_users.to_list())

print(f"Distinct src human users:  {src_users.n_unique():,}")
print(f"Distinct dest human users: {dest_users.n_unique():,}")
print(f"Union (src ∪ dest):        {len(all_users):,}")

In [ ]:
# Machine Counts
src_machines = (
    df
    .filter(pl.col("src_user").str.contains(r"\$"))
    .select(pl.col("src_user").unique())
    .collect()
    .to_series()
)

dest_machines = (
    df
    .filter(pl.col("dest_user").str.contains(r"\$"))
    .select(pl.col("dest_user").unique())
    .collect()
    .to_series()
)

all_machines = set(src_machines.to_list()) | set(dest_machines.to_list())

print(f"Distinct src machine accounts:  {src_machines.n_unique():,}")
print(f"Distinct dest machine accounts: {dest_machines.n_unique():,}")
print(f"Union (src ∪ dest):             {len(all_machines):,}")

In [ ]:
# Computer Counts
src_set = (
    df.select(pl.col("src_comp").unique())
      .collect()
      .to_series()
)

dest_set = (
    df.select(pl.col("dest_comp").unique())
      .collect()
      .to_series()
)

all_computers = set(src_set.to_list()) | set(dest_set.to_list())

print(f"Distinct source computers:      {src_set.n_unique():,}")
print(f"Distinct destination computers: {dest_set.n_unique():,}")
print(f"Distinct computers (union):     {len(all_computers):,}")

In [ ]:
# Proportion of events for machine/user/other
event_proportions = (
    df
    .select(
        total=pl.len(),
        machine_events=pl.col("src_user").str.contains(r"\$").sum(),
        user_events=pl.col("src_user").str.starts_with("U").sum(),
    )
    .with_columns(
        other_events=pl.col("total") - pl.col("machine_events") - pl.col("user_events"),
    )
    .with_columns(
        machine_pct=(pl.col("machine_events") / pl.col("total") * 100).round(2),
        user_pct=(pl.col("user_events") / pl.col("total") * 100).round(2),
        other_pct=(pl.col("other_events") / pl.col("total") * 100).round(2),
    )
    .collect()
)

event_proportions

In [ ]:
user_failure = (
    df
    .filter(pl.col("src_user").str.starts_with("U"))
    .with_columns(is_fail=(pl.col("outcome") == "Fail").cast(pl.Int64))
    .group_by("src_user")
    .agg(
        n_auths=pl.len(),
        n_fails=pl.col("is_fail").sum(),
    )
    .with_columns(failure_rate=pl.col("n_fails") / pl.col("n_auths"))
    .collect(streaming=True)     
)

user_failure.describe()

In [ ]:
machine_failure = (
    df
    .filter(pl.col("src_user").str.contains(r"\$"))      
    .with_columns(is_fail=(pl.col("outcome") == "Fail").cast(pl.Int64))
    .group_by("src_user")
    .agg(
        n_auths=pl.len(),
        n_fails=pl.col("is_fail").sum(),
    )
    .with_columns(failure_rate=pl.col("n_fails") / pl.col("n_auths"))
    .collect(streaming=True)
)

machine_failure.describe()

In [ ]:
# --- The only thing you change: how much time to show ---
window_seconds = 604800        

bucket_seconds = max(1, window_seconds // 100)   # auto-scale resolution to the window

ts = (
    df
    .filter(pl.col("time") < window_seconds)
    .with_columns(
        bucket=(pl.col("time") // bucket_seconds).cast(pl.Int64),
        group=pl.when(pl.col("src_user").str.contains(r"\$"))
                .then(pl.lit("machine"))
                .when(pl.col("src_user").str.starts_with("U"))
                .then(pl.lit("human"))
                .otherwise(pl.lit("other")),
    )
    .group_by(["group", "bucket"])
    .agg(n_auths=pl.len())
    .collect(streaming=True)
)

fig, ax = plt.subplots(figsize=(14, 5))
for grp in ["human", "machine", "other"]:
    sub = ts.filter(pl.col("group") == grp).sort("bucket")
    counts = sub["n_auths"].to_numpy()
    ax.plot(sub["bucket"].to_numpy() * bucket_seconds,
            counts / counts.max() if counts.max() else counts,
            label=grp, linewidth=1)
ax.set_xlabel("Seconds from start")
ax.set_ylabel("Auths per bucket (normalized to each group's peak)")
ax.set_title("Human vs machine — normalized shape (burstiness)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
user_dest = (
    df
    .filter(pl.col("src_user").str.starts_with("U"))
    .group_by("src_user")
    .agg(n_distinct_dest=pl.col("dest_comp").n_unique())
    .collect(streaming=True)
)

import numpy as np

vals = user_dest["n_distinct_dest"].to_numpy()
vals_pos = vals[vals > 0]   # log bins need positive values

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(vals_pos, bins=np.logspace(0, np.log10(vals_pos.max()), 50))
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Distinct destination computers per user (log)")
ax.set_ylabel("Number of users (log)")
ax.set_title("Distribution of distinct destinations per user (log-log)")
plt.tight_layout()
plt.show()

## Auth type, logon type, auth orientation

In [ ]:
auth_type_counts = (df.group_by("auth_type")
                      .agg(count=pl.len())
                      .sort("count", descending=True)
                      .collect(streaming=True))
print(auth_type_counts)

In [ ]:
auth_type_counts_users = (df.filter(pl.col("src_user").str.starts_with("U"))
                            .group_by("auth_type")
                            .agg(count=pl.len())
                            .sort("count", descending=True)
                            .collect(streaming=True))
print(auth_type_counts_users)

In [ ]:
logon_type_counts = (df.group_by("logon_type")
                      .agg(count=pl.len())
                      .sort("count", descending=True)
                      .collect(streaming=True))
print(logon_type_counts)

In [ ]:
auth_orientation = (df.group_by("auth_orientation")
                      .agg(count=pl.len())
                      .sort("count", descending=True)
                      .collect(streaming=True))
print(auth_orientation)

In [ ]:
auth_orientation_counts_users = (df.filter(pl.col("src_user").str.starts_with("U"))
                            .group_by("auth_orientation")
                            .agg(count=pl.len())
                            .sort("count", descending=True)
                            .collect(streaming=True))
print(auth_orientation_counts_users)